<a href="https://colab.research.google.com/github/natenorris10/MAT-422/blob/main/Homework_1_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Linear Algebra HW 1.4

In [1]:
import numpy as np

###1.4.1 - Singular Value Decomposition

For any $m \times n$ matrix $A,$ we know the matrix $A^TA$ is symmetric.  Because of this, it can be orthogonally diagonalized.  We're guaranteed to have all real eigenvalues, as well as all of them being nonnegative.  This can be shown by letting $\lambda$ be an eignevalue of $A^TA$ with corresponding unit eigenvector $\mathbf{v}.$  Then,

$0 \leq ||A\mathbf{v}||^2 = (A\mathbf{v}) \cdot (A\mathbf{v})
 = (A\mathbf{v})^T A\mathbf{v} = \mathbf{v}^T A^T A\mathbf{v}
 = \mathbf{v}^T \lambda \mathbf{v} = \lambda(\mathbf{v} \cdot \mathbf{v})
 = \lambda ||\mathbf{v}||^2 = \lambda$

This takes us to the formal definition of singular values:  If $A$ is an $m \times n$ matrix, the singular values of A are the square roots of the eigenvalues of $A^TA$ and are denoted by $\sigma_1, \dots, \sigma_n.$  Typical convention arranges the sigular values so that $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_n.$

In [ ]:
# Quick Example of Singular Values

mat1 = np.array([[1, 1],
                 [1, 0],
                 [0, 1]])

mat1Tmat1 = mat1.T @ mat1
print(mat1Tmat1)

lam1, lam2 = np.linalg.eigvals(mat1Tmat1)
sing1 = np.sqrt(lam1)
sing2 = np.sqrt(lam2)
print(sing1, sing2)

[[2 1]
 [1 2]]
1.7320508075688772 1.0


For Singular Value Decomposition, let $A$ be an $m \times n$ matrix with sigular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0$ and $\sigma_{r+1} = \sigma_{r+2} = \cdots = \sigma_n = 0.$  Then there exists an $m \times m$ orthogonal matrix $U,$ an $n \times n$ orthogonal matrix $V,$ and an $m \times n$ matrix $\Sigma,$ where
$\Sigma = \begin{pmatrix} D & O \\ O & O \end{pmatrix}$,
$D = \begin{pmatrix} \sigma_1 & \cdots & 0 \\ \vdots & \ddots & \vdots \\ 0 & \cdots & \sigma_r \end{pmatrix}$, and $O$ is a zero matrix of appropriate size, such that $A = U \Sigma V^T.$

In [ ]:
# Basic Example of Singular Value Decomposition of a matrix
mat2 = np.array([[1, 1, 0],
                 [0, 0, 1]])

mat2Tmat2 = mat2.T @ mat2
eigvals, eigvecs = np.linalg.eig(mat2Tmat2)

idx = np.argsort(eigvals)[::-1]
eigvals_sorted = eigvals[idx]
eigvecs_sorted = eigvecs[:, idx]

eval1, eval2, eval3 = eigvals_sorted
sigma1, sigma2, sigma3 = np.sqrt(eigvals_sorted)

V = eigvecs_sorted
Sigma = np.array([[sigma1, 0, 0],
                  [0, sigma2, 0]])

uvec1 = (1/sigma1)*(mat2 @ V.T[0])
uvec2 = (1/sigma2)*(mat2 @ V.T[1])
U = np.column_stack([uvec1, uvec2])

result = U @ Sigma @ V.T
print(f"Our initial matrix = \n{mat2}\nThe result of U @ Sigma @ V.T = \n{result}")

Our initial matrix = 
[[1 1 0]
 [0 0 1]]
The result of U @ Sigma @ V.T = 
[[1. 1. 0.]
 [0. 0. 1.]]


Singular value decomposition plays a key role in the next topic

###1.4.2 - Low-Rank Matrix Approximations

Let's say we wanted to have an approximation to a particular matrix.  What we can do is assign some value $k$ such that $k < r$, where r is the rank of our matrix.  If we decompose the matrix by SVD, we can approximate the matrix by taking only the first k values in the sum
$A = \sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T$, and thus change the sum to
$A_k = \sum_{j=1}^k \sigma_j \mathbf{u}_j \mathbf{v}_j^T$.

Then, with any matrix $B \in \mathbb{R}^{m \times n},$ we can conclude $||A-A_k||_2 \le ||A - B||_2, $ where the subscript $2$ implies matrices in the $l_2$ norm.

In [ ]:
# Basic Example of Low Rank Matrix Approximation

mat3 = np.array([[1, 2, 3],
                 [4, 5, 6],
                 [7, 8, 10],
                 [1, 0, 1]])

U, S, Vt = np.linalg.svd(mat3)

k = 1
mat3_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

u = np.array([1, 2, 3, 0]).reshape(-1, 1)
v = np.array([1, 1, 1]).reshape(1, -1)
matB = u @ v

error_mat3_k = np.linalg.norm(mat3 - mat3_k)
error_matB = np.linalg.norm(mat3 - matB)

print(f"{error_mat3_k} <= {error_matB}")

1.210885334831258 <= 11.224972160321824


###1.4.3 - Principal Component Analysis

The main idea behind PCA is to use correlations in our data to find new patterns.  We first center our data by putting it in mean-deviation form, and find the directions of maximum variance from a covariance matrix.  Then we take the eigenvalues and eigenvectors, with each eigenvalue describing the level of variance related to its particular eigenvector.  We then multiply our data by the largest eigenvectors (the top components) which projects it into a lower dimension.

In [3]:
# Basic example of PCA

A = np.array([[2.5, 2.4],
              [0.5, 0.7],
              [2.2, 2.9],
              [1.9, 2.2],
              [3.1, 3.0]])

mean = np.mean(A, axis=0)
centered_A = A - mean

cov_mat = np.cov(centered_A, rowvar=False)

eigenvalues, eigenvectors = np.linalg.eig(cov_mat)

order = np.argsort(eigenvalues[::-1])
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

top_pc = eigenvectors[:, 0].reshape(-1, 1)
A_proj = centered_A @ top_pc

print(f"The original data was\n{A}\nThe resulting data after PCA is \n{A_proj}")

The original data was
[[2.5 2.4]
 [0.5 0.7]
 [2.2 2.9]
 [1.9 2.2]
 [3.1 3. ]]
The resulting data after PCA is 
[[ 0.44362444]
 [-2.17719404]
 [ 0.57071239]
 [-0.12902465]
 [ 1.29188186]]
